# End-to-End ML Project — Week 3
## Data Preprocessing & Feature Engineering

**Dataset:** Vehicle Sales Data (car_prices_sample.csv)  
**Goal:** Clean data, encode features, scale numerical columns, and save processed dataset.

In [2]:
# Download dataset via curl
import subprocess, os
os.makedirs('../data/processed', exist_ok=True)
if not os.path.exists('../data/car_prices_sample.csv'):
    subprocess.run(['curl', '-L', '-o', '../data/car_prices_sample.csv',
                    'https://raw.githubusercontent.com/placeholder/sample_1.csv'])
print('Data file ready.')

Data file ready.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

raw_df = pd.read_csv('../data/car_prices_sample.csv')
print(f'Raw dataset shape: {raw_df.shape}')
raw_df.head()

Raw dataset shape: (50000, 15)


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,sellingprice,saledate
0,2014,Kia,Optima,LX,Sedan,automatic,5xxgm4a71eg328703,tx,49.0,11485.0,gray,gray,kia motors america inc,15700,Wed Feb 04 2015 02:30:00 GMT-0800 (PST)
1,2014,Kia,Optima,LX,Sedan,automatic,5xxgm4a71eg318141,tx,26.0,14265.0,white,beige,kia motors america inc,15900,Wed Feb 04 2015 02:30:00 GMT-0800 (PST)
2,2006,Chrysler,Town and Country,Limited,Minivan,automatic,2a8gp64l96r821834,az,19.0,92649.0,gold,tan,onesource/southwest remarketing,4700,Thu Jan 22 2015 03:00:00 GMT-0800 (PST)
3,2006,Chevrolet,Silverado 1500,LS,Extended Cab,NaN,1gcek19bx6z269153,tx,22.0,166999.0,white,gray,texas direct auto,6700,Wed Feb 25 2015 02:20:00 GMT-0800 (PST)
4,2013,Chevrolet,Black Diamond Avalanche,LT,Crew Cab,automatic,3gntkfe78dg263779,tn,43.0,29265.0,—,black,north lily auto sales llc,32250,Wed Jan 28 2015 02:30:00 GMT-0800 (PST)


## 1. Column Type Analysis

In [4]:
print('=== NUMERICAL COLUMNS ===')
num_cols = raw_df.select_dtypes(include='number').columns.tolist()
print(num_cols)

print('\n=== CATEGORICAL COLUMNS (object dtype) ===')
cat_cols = raw_df.select_dtypes(include='object').columns.tolist()
print(cat_cols)

print('\n=== CARDINALITY ANALYSIS ===')
for col in cat_cols:
    print(f'  {col:15s}: {raw_df[col].nunique():5d} unique values — ', end='')
    if raw_df[col].nunique() <= 15:
        print('→ One-Hot Encoding')
    elif raw_df[col].nunique() <= 100:
        print('→ One-Hot or Label Encoding')
    else:
        print('→ Hash Encoding (high cardinality)')

=== NUMERICAL COLUMNS ===
['year', 'condition', 'odometer', 'sellingprice']

=== CATEGORICAL COLUMNS (object dtype) ===
['make', 'model', 'trim', 'body', 'transmission', 'vin', 'state', 'color', 'interior', 'seller', 'saledate']

=== CARDINALITY ANALYSIS ===
  make           :    78 unique values — → One-Hot or Label Encoding
  model          :   741 unique values — → Hash Encoding (high cardinality)
  trim           :  1211 unique values — → Hash Encoding (high cardinality)
  body           :    71 unique values — → One-Hot or Label Encoding
  transmission   :     3 unique values — → One-Hot Encoding
  vin            : 49943 unique values — → Hash Encoding (high cardinality)
  state          :    41 unique values — → One-Hot or Label Encoding
  color          :    23 unique values — → One-Hot or Label Encoding
  interior       :    17 unique values — → One-Hot or Label Encoding
  seller         :  5354 unique values — → Hash Encoding (high cardinality)
  saledate       :  2306 unique 

## 2. Full Preprocessing Pipeline

| Column | Type | Strategy |
|--------|------|----------|
| `year` | Numerical | Keep as-is + StandardScaler |
| `condition` | Numerical | Median imputation + StandardScaler |
| `odometer` | Numerical | Median imputation, IQR cap + StandardScaler |
| `make` | Categorical (low card.) | Capitalize → One-Hot Encoding |
| `body` | Categorical (low card.) | Lowercase, group rare → One-Hot Encoding |
| `transmission` | Categorical (binary) | Lowercase, drop error row → One-Hot |
| `state` | Categorical (med. card.) | One-Hot Encoding |
| `color` | Categorical (med. card.) | One-Hot Encoding |
| `interior` | Categorical (med. card.) | One-Hot Encoding |
| `model` | Categorical (high card.) | Hash Encoding (n_components=32) |
| `trim` | Categorical (high card.) | Hash Encoding (n_components=16) |
| `vin` | ID column | Drop |
| `seller` | High card., not informative | Drop |
| `saledate` | Raw string datetime | Drop |

In [5]:
from category_encoders import OneHotEncoder, HashingEncoder

def preprocess(raw_df: pd.DataFrame, fit: bool = True,
               scaler: StandardScaler = None,
               ohe=None, hash_model=None, hash_trim=None) -> tuple:
    """
    Full preprocessing pipeline for vehicle sales dataset.
    
    Parameters
    ----------
    raw_df : Raw dataframe
    fit    : True when fitting on training data, False for inference
    scaler, ohe, hash_model, hash_trim : pre-fitted transformers for inference mode
    
    Returns
    -------
    (X_processed, y, scaler, ohe, hash_model, hash_trim)
    """
    df = raw_df.copy()
    
    # --- Drop non-informative columns ---
    drop_cols = [c for c in ['vin', 'seller', 'saledate'] if c in df.columns]
    df = df.drop(columns=drop_cols)
    
    # --- Target ---
    y = df['sellingprice'].copy()
    df = df.drop(columns=['sellingprice'])
    
    # --- Step 1: Remove data entry error in transmission ---
    if 'transmission' in df.columns:
        df = df[df['transmission'] != 'Sedan']
        y = y[df.index]  # keep y aligned
        df = df.reset_index(drop=True)
        y = y.reset_index(drop=True)
    
    # --- Step 2: Price outlier filtering (only in fit mode) ---
    if fit:
        p99 = np.percentile(y, 99)
        mask = y <= p99
        df = df[mask].reset_index(drop=True)
        y = y[mask].reset_index(drop=True)
        print(f'Price outliers removed (>${p99:,.0f}): {(~mask).sum()} rows')
    
    # --- Step 3: Text cleaning ---
    str_cols = df.select_dtypes(include='object').columns
    df[str_cols] = df[str_cols].apply(lambda x: x.str.lower().str.strip())
    
    # make: capitalize
    if 'make' in df.columns:
        df['make'] = df['make'].str.capitalize()
    
    # body: group rare types
    if 'body' in df.columns:
        if fit:
            body_counts = df['body'].value_counts()
            rare_bodies = body_counts[body_counts < 50].index
        df['body'] = df['body'].replace(rare_bodies, 'other')
    
    # --- Step 4: Impute missing values ---
    # Numerical: median
    num_impute_cols = ['condition', 'odometer', 'year']
    num_impute_cols = [c for c in num_impute_cols if c in df.columns]
    if fit:
        num_imp = SimpleImputer(strategy='median')
        df[num_impute_cols] = num_imp.fit_transform(df[num_impute_cols])
    else:
        df[num_impute_cols] = num_imp.transform(df[num_impute_cols])
    
    # Categorical: fill 'unknown'
    cat_impute_cols = [c for c in df.select_dtypes('object').columns]
    df[cat_impute_cols] = df[cat_impute_cols].fillna('unknown')
    
    # --- Step 5: IQR cap on odometer ---
    if 'odometer' in df.columns:
        if fit:
            Q1 = df['odometer'].quantile(0.25)
            Q3 = df['odometer'].quantile(0.75)
            IQR = Q3 - Q1
            odo_lower = Q1 - 1.5 * IQR
            odo_upper = Q3 + 1.5 * IQR
        df['odometer'] = df['odometer'].clip(lower=odo_lower, upper=odo_upper)
    
    # --- Step 6: Feature Engineering ---
    # Age feature (more intuitive than raw year)
    df['age'] = 2026 - df['year']
    # Interaction: older car with high mileage penalized more
    df['age_x_odometer'] = df['age'] * df['odometer'] / 1e6
    
    # --- Step 7: Scale numerical features ---
    scale_cols = ['year', 'condition', 'odometer', 'age', 'age_x_odometer']
    scale_cols = [c for c in scale_cols if c in df.columns]
    
    if fit:
        scaler = StandardScaler()
        df[scale_cols] = scaler.fit_transform(df[scale_cols])
    else:
        df[scale_cols] = scaler.transform(df[scale_cols])
    
    # --- Step 8: Encode categorical features ---
    ohe_cols = ['make', 'body', 'transmission', 'state', 'color', 'interior']
    ohe_cols = [c for c in ohe_cols if c in df.columns]
    
    hash_model_cols = ['model'] if 'model' in df.columns else []
    hash_trim_cols = ['trim'] if 'trim' in df.columns else []
    
    if fit:
        ohe = OneHotEncoder(cols=ohe_cols, use_cat_names=True, handle_unknown='ignore')
        df = ohe.fit_transform(df)
        
        hash_model = HashingEncoder(cols=hash_model_cols, n_components=32) if hash_model_cols else None
        if hash_model:
            df = hash_model.fit_transform(df)
        
        hash_trim = HashingEncoder(cols=hash_trim_cols, n_components=16) if hash_trim_cols else None
        if hash_trim:
            df = hash_trim.fit_transform(df)
    else:
        df = ohe.transform(df)
        if hash_model:
            df = hash_model.transform(df)
        if hash_trim:
            df = hash_trim.transform(df)
    
    print(f'Processed shape: {df.shape}')
    print(f'Missing values: {df.isnull().sum().sum()}')
    
    return df, y, scaler, ohe, hash_model, hash_trim


X_processed, y, scaler, ohe, hash_model, hash_trim = preprocess(raw_df, fit=True)
X_processed.head()

ModuleNotFoundError: No module named 'category_encoders'

## 3. Visualise Processed Features

In [5]:
# Distribution of scaled numerical features
scale_cols_check = ['year', 'condition', 'odometer', 'age']
scale_cols_check = [c for c in scale_cols_check if c in X_processed.columns]

fig, axes = plt.subplots(1, len(scale_cols_check), figsize=(16, 4))
for ax, col in zip(axes, scale_cols_check):
    sns.histplot(X_processed[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(f'{col} (scaled)')
    ax.axvline(0, color='red', linestyle='--', alpha=0.7)
plt.suptitle('Scaled Numerical Feature Distributions (mean≈0, std≈1)', y=1.02)
plt.tight_layout()
plt.savefig('../reports/scaled_features.png', dpi=100, bbox_inches='tight')
plt.show()

# Verify scaling
print('Mean of scaled features (should be ~0):')
print(X_processed[scale_cols_check].mean().round(3))
print('\nStd of scaled features (should be ~1):')
print(X_processed[scale_cols_check].std().round(3))

NameError: name 'X_processed' is not defined

In [ ]:
# Feature correlation with target after preprocessing
# Compute correlations for the numerical processed features with target
num_features_processed = X_processed.select_dtypes(include='number').columns[:20]
corr_with_target = X_processed[num_features_processed].corrwith(y).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
corr_with_target.head(20).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Features — Absolute Correlation with Selling Price')
ax.set_xlabel('|Pearson r|')
plt.tight_layout()
plt.savefig('../reports/feature_correlations.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Save Processed Data

In [1]:
from sklearn.model_selection import train_test_split
import joblib
import os

os.makedirs('../data/processed', exist_ok=True)

# Train/test split BEFORE saving to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

# Save processed datasets
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save fitted transformers
joblib.dump(scaler, '../data/processed/scaler.joblib')
joblib.dump(ohe, '../data/processed/ohe.joblib')
if hash_model:
    joblib.dump(hash_model, '../data/processed/hash_model.joblib')
if hash_trim:
    joblib.dump(hash_trim, '../data/processed/hash_trim.joblib')

print('Saved files:')
print(f'  X_train: {X_train.shape}')
print(f'  X_test:  {X_test.shape}')
print(f'  y_train: {y_train.shape}')
print(f'  y_test:  {y_test.shape}')
print(f'\nTotal features: {X_train.shape[1]}')

NameError: name 'X_processed' is not defined